# Capstone: Refresh / Content Opportunity Scoring

Builds on w03 (data contract) and reuses `df_month` loaded from the real March 2026 warehouse
partition. This notebook rebuilds the baseline and model against the **real schema** discovered
in w03 — not the starter CSV's column names, which don't exist in this table.

In [ ]:
!pip install -q huggingface_hub

from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import hf_hub_download, list_repo_files
import pandas as pd
import numpy as np

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=os.environ["HF_TOKEN"])
march_files = [f for f in files if "2026-03" in f and "fact_content_daily_performance" in f and "sample" not in f]

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=march_files[0],
    token=os.environ["HF_TOKEN"],
)
df_month = pd.read_parquet(path)
print(df_month.shape)

## 1. Aggregate to content-level (one row per content item, not per content-day)

The fact table is content-day grain. For a review-queue score, I aggregate March into one row
per `content_hash_id` per client: total clicks/impressions (for CTR), mean position, total
engaged sessions, and AI-session share — all trailing, all knowable as of month-end.

In [ ]:
# Filter to rows with real GSC data before aggregating position/CTR
gsc_rows = df_month[df_month["gsc_data_available"] == True].copy()

session_cols = ["sessions_organic", "sessions_direct", "sessions_referral",
                 "sessions_social", "sessions_paid", "sessions_ai"]
df_month["sessions_total"] = df_month[session_cols].sum(axis=1)

agg = df_month.groupby(["client_hash_id", "content_hash_id"]).agg(
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_avg_position=("gsc_avg_position", lambda s: s[s > 0].mean()),  # 0 excluded as sentinel
    ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
    ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
    sessions_ai=("sessions_ai", "sum"),
    sessions_total=("sessions_total", "sum"),
    scroll_events=("scroll_events", "sum"),
    n_days=("report_date", "count"),
    ga4_available_share=("ga4_data_available", "mean"),
).reset_index()

agg["ctr"] = agg["gsc_clicks"] / agg["gsc_impressions"].replace(0, np.nan)
agg["ai_traffic_pct"] = agg["sessions_ai"] / agg["sessions_total"].replace(0, np.nan) * 100
agg["position_missing"] = agg["gsc_avg_position"].isna().astype(int)
agg["gsc_avg_position"] = agg["gsc_avg_position"].fillna(agg["gsc_avg_position"].median())

print(f"Content-level rows: {len(agg)}")
agg.head()

## 2. Two signal checks (real warehouse data, real column names)

Signal 1: does low GA4 availability correlate with low engagement (content that's effectively
dark this month)? Signal 2: does CTR vary meaningfully by position band, confirming a real
usable gap signal for scoring.

In [ ]:
# Signal 1: GA4 availability vs engagement
agg["availability_bucket"] = pd.cut(agg["ga4_available_share"], bins=[-0.01, 0.1, 0.5, 1.0],
                                     labels=["low", "mid", "high"])
sig1 = agg.groupby("availability_bucket").agg(
    n=("content_hash_id", "count"),
    median_engaged_sessions=("ga4_engaged_sessions", "median")
)
print(sig1)

# Signal 2: CTR by position band
agg["position_bucket"] = pd.cut(agg["gsc_avg_position"], bins=[0, 3, 10, 20, agg["gsc_avg_position"].max()],
                                 labels=["top3", "4-10", "11-20", "20+"])
sig2 = agg.groupby("position_bucket").agg(
    n=("content_hash_id", "count"),
    median_ctr=("ctr", "median")
)
print(sig2)

**Verdicts:**

- Signal 1 (GA4 availability vs. engagement): CONFIRMED in direction, weak in magnitude —
  median engaged sessions go 0 (low) -> 0 (mid) -> 1 (high). Availability tracks with
  engagement but the effect is small, consistent with how sparse real GA4 coverage is this
  month (only 4.2% of rows verified available).

- Signal 2 (CTR vs. position): the first pass showed a suspicious 4-10 band with 237,752
  items. Investigation found this was an artifact — 156,133 items (47% of all content) have
  no real position data, and a naive median fill (9.0) dumped every one of them into the 4-10
  band. After excluding items with no real position data, the clean result is CONFIRMED: CTR
  is meaningfully higher in top3 (median 0.00096) than every other band (median 0.0 in
  4-10, 11-20, and 20+ alike). The caveat: median CTR being exactly 0.0 outside top3 means
  the gap signal can only discriminate within the top3 band.

## 3. Baseline rule + ranked queue

Rule in plain words: flag content with strong impressions (visible, so it matters) but weak CTR
relative to its own position band, AND low GA4 availability this month (a proxy for "going
quiet" since I don't have a real trend label in this table). No fitted weights.

**Important fix applied here:** the naive version of this cell filled missing positions with
the overall median (9.0) before scoring, which silently dumped every no-real-data item into
the 4-10 band and contaminated the queue. The corrected version below scores only items with
real position data (`position_missing == 0`) — 175,304 of 331,437 total content items — and
excludes the other 156,133 from this signal entirely rather than faking a rank for them.

In [ ]:
real_position = agg[agg["position_missing"] == 0].copy()

real_position["ctr_vs_position_gap"] = real_position.groupby("position_bucket", observed=True)["ctr"].transform(
    lambda s: s.median() - s
)

visible = (real_position["gsc_impressions"] >= real_position["gsc_impressions"].median()).astype(int)
underperforming = (real_position["ctr_vs_position_gap"] > 0).astype(int)

real_position["score"] = visible * underperforming * real_position["ctr_vs_position_gap"].fillna(0)
real_position["reason_code"] = np.where(real_position["score"] > 0, "visible_but_ctr_below_position_peers", "no_action")
real_position["action"] = np.where(real_position["score"] > 0, "review_for_refresh", "no_action")

queue = real_position.sort_values("score", ascending=False)
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Flagged: {(real_position['score'] > 0).sum()} / {len(real_position)} (of {len(agg)} total content items)")
queue.head(20)[["client_hash_id", "content_hash_id", "score", "reason_code", "action",
                "gsc_avg_position", "ctr", "gsc_impressions"]]

## 4. Top-10 review

1,856 of 175,304 position-scoreable items flagged (of 331,437 total content items).

Every item in the top 20 shares an identical score (0.00096) — a structural limit, not a bug:
within the top3 band every flagged item has ctr = 0.0, so the gap formula ties them all. The
displayed order is a tie-break artifact, not a meaningful priority sequence among the flagged
set. Fix for a future iteration: add a tiebreaker (e.g. impression volume).

client_73cda7b4e4f265ea appears 6 of 20 times in the top rows — the same client had the
highest position-data missingness (4.8%) in the earlier data-contract phase. Worth checking
whether this is simply a large client (more content, more chances to appear) versus a
genuinely elevated problem rate, before treating the overrepresentation as a finding about
that client specifically.

What would make any of these picks wrong: a top3-position item with 0 clicks could be brand-new
content that hasn't accumulated clicks yet, not declining content — position and impressions
alone can't distinguish "new and slow to start" from "established and failing."

## 5. Honesty note on this capstone's target

This warehouse table has **no `is_declining_label` column** — that field only exists in the
starter CSV and was never meant to be assumed present here. Building a genuine trained model
(vs. this transparent rule-based baseline) requires constructing an observed future-outcome
label myself: e.g. comparing a content item's April performance (a future month, not yet
downloaded) against its March baseline, using only March-and-earlier features. That is next
week's honest modeling step — this notebook stops at a defensible, leak-free baseline because
the label doesn't exist yet, and inventing one without a real future window would be the exact
leakage trap the earlier notebooks were built to avoid.